First Pattern - Interrupt Before

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage
from typing import TypedDict, Annotated

class State(TypedDict):
    messages: Annotated[list, add_messages]

def draft_email(state: State) -> dict:
    """Draft an email — this runs automatically."""
    return {"messages": ["DRAFT: Dear Boss, I'd like to request a raise based on my performance..."]}

def send_email(state: State) -> dict:
    """Send the email — we want human approval first!"""
    return {"messages": ["✅ Email sent successfully!"]}

# Build graph
builder = StateGraph(State)
builder.add_node("draft", draft_email)
builder.add_node("send", send_email)
builder.add_edge(START, "draft")
builder.add_edge("draft", "send")
builder.add_edge("send", END)

memory = InMemorySaver()

# Compile with interrupt_before — graph pauses BEFORE "send" node runs
graph = builder.compile(
    checkpointer=memory,
    interrupt_before=["send"]           

config = {"configurable": {"thread_id": "email_1"}}

# Run — will execute "draft" then PAUSE before "send"
result = graph.invoke(
    {"messages": [HumanMessage(content="Write an email asking for a raise")]},
    config=config
)

print("📝 Draft:", result["messages"][-1].content)
print("\nDo you approve sending this? (y/n)")
approval = input("> ")

if approval.lower() == "y":
    # Resume — pass None to continue from where we left off
    result = graph.invoke(None, config=config)
    print(result["messages"][-1].content)   
else:
    print("❌ Email cancelled by human.")

📝 Draft: DRAFT: Dear Boss, I'd like to request a raise based on my performance...

Do you approve sending this? (y/n)
❌ Email cancelled by human.


Second Pattern - Interrupt After

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage
from typing import TypedDict, Annotated

class State(TypedDict):
    messages: Annotated[list, add_messages]

def draft_email(state: State) -> dict:
    """Draft an email — this runs automatically."""
    return {"messages": ["DRAFT: Dear Boss, I'd like to request a raise based on my performance..."]}

def send_email(state: State) -> dict:
    """Send the email — we want human approval first!"""
    return {"messages": ["✅ Email sent successfully!"]}

# Build graph
builder = StateGraph(State)
builder.add_node("draft", draft_email)
builder.add_node("send", send_email)
builder.add_edge(START, "draft")
builder.add_edge("draft", "send")
builder.add_edge("send", END)

memory = InMemorySaver()

# Compile with interrupt_before — graph pauses BEFORE "send" node runs
graph = builder.compile(
    checkpointer=memory,
    interrupt_after=["draft"]   
)        

config = {"configurable": {"thread_id": "email_1"}}

# Run — will execute "draft" then PAUSE before "send"
result = graph.invoke(
    {"messages": [HumanMessage(content="Write an email asking for a raise")]},
    config=config
)

print("📝 Draft:", result["messages"][-1].content)
print("\nDo you approve sending this? (y/n)")
approval = input("> ")

if approval.lower() == "y":
    # Resume — pass None to continue from where we left off
    result = graph.invoke(None, config=config)
    print(result["messages"][-1].content)   
else:
    print("❌ Email cancelled by human.")

Third Pattern - Interrupt Function

In [6]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.types import interrupt, Command
from langchain_core.messages import HumanMessage
from typing import TypedDict, Annotated

class State(TypedDict):
    messages: Annotated[list, add_messages]

def draft_email(state: State) -> dict:
    return {"messages": ["DRAFT: Dear Boss, I'd like to request a raise based on my performance..."]}

def review_email(state: State) -> dict:
    response = interrupt({"question": "Approve this email?", "options": "yes / no"})
    if response.lower() in ["yes", "y"]:
        return {"messages": ["✅ Email sent successfully!"]}
    return {"messages": [f"❌ Email cancelled. Reason: {response}"]}

builder = StateGraph(State)
builder.add_node("draft", draft_email)
builder.add_node("review", review_email)
builder.add_edge(START, "draft")
builder.add_edge("draft", "review")
builder.add_edge("review", END)

graph = builder.compile(checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "email_1"}}

result = graph.invoke({"messages": [HumanMessage(content="Write a raise email")]}, config)
print("📝 Draft:", result["messages"][-1].content)

approval = input("Approve? (yes/no): ")
result = graph.invoke(Command(resume=approval), config)
print(result["messages"][-1].content)

📝 Draft: DRAFT: Dear Boss, I'd like to request a raise based on my performance...
❌ Email cancelled. Reason: n


Fourth Pattern - Update State

In [9]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage
from typing import TypedDict, Annotated

class State(TypedDict):
    messages: Annotated[list, add_messages]

def draft_email(state: State) -> dict:
    return {"messages": ["DRAFT: Dear Boss, I'd like to request a raise based on my performance..."]}

def send_email(state: State) -> dict:
    return {"messages": ["✅ Email sent successfully!"]}

builder = StateGraph(State)
builder.add_node("draft", draft_email)
builder.add_node("send", send_email)
builder.add_edge(START, "draft")
builder.add_edge("draft", "send")
builder.add_edge("send", END)

graph = builder.compile(checkpointer=InMemorySaver(), interrupt_before=["send"])
config = {"configurable": {"thread_id": "email_1"}}

result = graph.invoke({"messages": [HumanMessage(content="Write a raise email")]}, config)
print("📝 Draft:", result["messages"][-1].content)

approval = input("Approve? Edit? (yes/edit/no): ")

if approval.lower() == "edit":
    # Inject a new message into the state before resuming
    graph.update_state(config, {
        "messages": [HumanMessage(content="Make it more polite and mention 3 years of service")]
    })
    result = graph.invoke(None, config)
    print(result["messages"][-1].content)

elif approval.lower() in ["yes", "y"]:
    result = graph.invoke(None, config)
    print(result["messages"][-1].content)

else:
    print("❌ Email cancelled.")

📝 Draft: DRAFT: Dear Boss, I'd like to request a raise based on my performance...
✅ Email sent successfully!
